In [36]:
from sklearn.model_selection import StratifiedKFold
import yaml
import numpy as np

In [37]:
with open('../config.yml', 'r') as file:
    config = yaml.safe_load(file)

resting_path = config['data']['resting']
photic_path = config['data']['photic']

In [38]:
import json
import pandas as pd
from pathlib import Path

def load_resting_bids_dataset(dataset_path):
    """
    Load resting-state EEG BIDS dataset.

    Returns:
        df,
        (control_subjects, ad_subjects, ftd_subjects)
    """

    dataset_dir = Path(dataset_path)

    eeg_suffixes = {".set", ".edf", ".bdf", ".eeg", ".vhdr", ".fif"}
    eeg_files = [
        f for f in dataset_dir.rglob("*_eeg.*")
        if f.is_file() and f.suffix.lower() in eeg_suffixes and "derivatives" not in f.parts
    ]

    def parse_bids_entities(eeg_file: Path) -> dict:
        entities = {"participant_id": None, "task": None}

        stem = eeg_file.name.rsplit(".", 1)[0]
        for token in stem.split("_"):
            if "-" not in token:
                continue
            key, value = token.split("-", 1)
            if key == "sub":
                entities["participant_id"] = f"sub-{value}"
            elif key == "task":
                entities["task"] = value

        if entities["participant_id"] is None:
            entities["participant_id"] = next(
                (p for p in eeg_file.parts if p.startswith("sub-")), None
            )

        return entities

    records = []

    for eeg_file in sorted(eeg_files):
        entities = parse_bids_entities(eeg_file)

        base = eeg_file.name.rsplit("_eeg", 1)[0]
        channels_tsv = eeg_file.with_name(f"{base}_channels.tsv")
        eeg_json = eeg_file.with_name(f"{base}_eeg.json")

        json_meta = {}
        if eeg_json.exists():
            with open(eeg_json, "r", encoding="utf-8") as f:
                json_meta = json.load(f)

        records.append({
            **entities,
            "eeg_file": str(eeg_file),
            "channels_file": str(channels_tsv) if channels_tsv.exists() else None,
            "eeg_json": str(eeg_json) if eeg_json.exists() else None,
            "file_format": eeg_file.suffix.lower().lstrip("."),
            "SamplingFrequency": json_meta.get("SamplingFrequency"),
        })

    df = pd.DataFrame(records)

    # Merge participant metadata
    participants_tsv = dataset_dir / "participants.tsv"
    if participants_tsv.exists() and not df.empty:
        participants_df = pd.read_csv(participants_tsv, sep="\t", encoding="utf-8-sig")
        participants_df.columns = [c.strip() for c in participants_df.columns]
        df = df.merge(participants_df, on="participant_id", how="left")

    if df.empty:
        return df, ([], [], [])

    unique_subjects = df["participant_id"].dropna().unique()

    control_subjects = []
    ad_subjects = []
    ftd_subjects = []

    for subject in unique_subjects:
        subject_df = df[df["participant_id"] == subject]
        group = subject_df["Group"].iloc[0] if "Group" in subject_df.columns else None

        if group == "C":
            control_subjects.append(subject)
        elif group == "A":
            ad_subjects.append(subject)
        elif group == "F":
            ftd_subjects.append(subject)

    print(f"Number of Control subjects: {len(control_subjects)}")
    print(f"Number of AD subjects: {len(ad_subjects)}")
    print(f"Number of FTD subjects: {len(ftd_subjects)}")

    df = df.sort_values(["participant_id", "task"]).reset_index(drop=True)

    return df, (control_subjects, ad_subjects, ftd_subjects)


In [39]:
resting_df, (controls, ad, ftd) = load_resting_bids_dataset(resting_path)

# Optional: view only the JSON metadata columnss
metadata_cols = [
    "participant_id",
    "task",
    "EEGPlacementScheme",
    "EEGReference",
    "CapManufacturersModelName",
    "CapManufacturer",
    "EEGChannelCount",
    "RecordingType",
    "SamplingFrequency",
]
#resting_df[metadata_cols].head()
resting_df.head()

Number of Control subjects: 29
Number of AD subjects: 36
Number of FTD subjects: 23


,participant_id,task,eeg_file,channels_file,eeg_json,file_format,SamplingFrequency,Gender,Age,Group,MMSE
0,sub-001,eyesclosed,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,set,500,F,57,A,16
1,sub-002,eyesclosed,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,set,500,F,78,A,22
2,sub-003,eyesclosed,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,set,500,M,70,A,14
3,sub-004,eyesclosed,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,set,500,F,67,A,20
4,sub-005,eyesclosed,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,/Users/buraa/Repo/TaskEEG-Dx/data/raw/ds004504...,set,500,M,70,A,22


In [42]:
split_seed = config.get('split_seed')
n_splits = 5
kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=split_seed)

# 1) Build subject list
all_subjects = np.concatenate([
    np.array(controls, dtype=object),
    np.array(ad, dtype=object),
    np.array(ftd, dtype=object),
])

# 2) Build labels (multiclass)
# 0 = Control, 1 = AD, 2 = FTD
all_labels = np.concatenate([
    np.full(len(controls), 0, dtype=int),
    np.full(len(ad), 1, dtype=int),
    np.full(len(ftd), 2, dtype=int),
])

label_name = {0: "Control", 1: "AD", 2: "FTD"}

# 3) Stratified subject-wise splits
split_counter = 0
for train_index, test_index in kf.split(all_subjects, all_labels):
    split_counter += 1

    train_subjects = all_subjects[train_index]
    test_subjects = all_subjects[test_index]

    train_labels = all_labels[train_index]
    test_labels = all_labels[test_index]

    print(f"\n=== Split {split_counter} ===")
    print("Train counts:",
          {label_name[k]: int(np.sum(train_labels == k)) for k in np.unique(all_labels)})
    print("Test counts: ",
          {label_name[k]: int(np.sum(test_labels == k)) for k in np.unique(all_labels)})
    print(f"Train labels: {[label_name[l] for l in train_labels]}")
    print(f"Train subjects: {[s for s in train_subjects]}")
    print(f"Test labels: {[label_name[l] for l in test_labels]}")
    print(f"Test subjects: {[s for s in test_subjects]}")


=== Split 1 ===
Train counts: {'Control': 23, 'AD': 29, 'FTD': 18}
Test counts:  {'Control': 6, 'AD': 7, 'FTD': 5}
Train labels: ['Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'Control', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'AD', 'FTD', 'FTD', 'FTD', 'FTD', 'FTD', 'FTD', 'FTD', 'FTD', 'FTD', 'FTD', 'FTD', 'FTD', 'FTD', 'FTD', 'FTD', 'FTD', 'FTD', 'FTD']
Train subjects: ['sub-037', 'sub-038', 'sub-039', 'sub-040', 'sub-041', 'sub-042', 'sub-043', 'sub-045', 'sub-046', 'sub-047', 'sub-048', 'sub-049', 'sub-054', 'sub-056', 'sub-057', 'sub-058', 'sub-059', 'sub-060', 'sub-061', 'sub-062', 'sub-063', 'sub-064', 'sub-065', 'sub-001', 'sub-002', 'sub-004', 'sub-006', 's